# Lung Segmentation — Preprocessing Pipeline v2
### 4-Class Unified Mask Schema
| Class | Value | Source |
|-------|-------|--------|
| Background | 0 | Both datasets |
| Lung | 1 | COVID LungMasks (binary PNG) + LUNA seg .mhd files |
| Nodule | 2 | LUNA annotations.csv |
| Infection (GGO / Consolidation) | 3 | COVID InfectionMasks (binary PNG) |

In [ ]:
! pip install SimpleITK -q

In [ ]:
import os
import glob
import numpy as np
import cv2
import pandas as pd
import SimpleITK as sitk
from tqdm import tqdm
import matplotlib.pyplot as plt

# ─────────────────────────────────────────────
# PATHS
# ─────────────────────────────────────────────
COVID_BASE          = '/kaggle/input/datasets/nguyentienda32143/covid-19-ct-lung-and-infection-segmentation/data/data'
COVID_IMG_DIR       = os.path.join(COVID_BASE, 'Covid-19-2')
COVID_INF_MASK_DIR  = os.path.join(COVID_BASE, 'InfectionMasks')
COVID_LUNG_MASK_DIR = os.path.join(COVID_BASE, 'LungMasks')

LUNA_BASE           = '/kaggle/input/datasets/fanbyprinciple/luna-lung-cancer-dataset'
# The seg-lungs-LUNA16 folder contains BOTH the raw CT .mhd files AND their
# paired lung segmentation .mhd files. We detect which is which at runtime.
LUNA_IMG_DIR        = os.path.join(LUNA_BASE, 'seg-lungs-LUNA16/seg-lungs-LUNA16')
LUNA_ANNOTATIONS    = os.path.join(LUNA_BASE, 'annotations.csv')

OUTPUT_IMG_DIR      = '/kaggle/working/processed_data/images'
OUTPUT_MASK_DIR     = '/kaggle/working/processed_data/masks'
os.makedirs(OUTPUT_IMG_DIR, exist_ok=True)
os.makedirs(OUTPUT_MASK_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# CLASS CONSTANTS
# ─────────────────────────────────────────────
CLASS_BACKGROUND = 0
CLASS_LUNG       = 1
CLASS_NODULE     = 2
CLASS_INFECTION  = 3
NUM_CLASSES      = 4

TARGET_SIZE      = (256, 256)

# Every Nth non-nodule slice from LUNA will be saved as a background example.
# N=10 means ~10% of clean slices are kept, giving the model negative examples
# without drowning out the rarer nodule slices.
BACKGROUND_SLICE_EVERY_N = 10

print("Configuration ready.")
print(f"  Target size              : {TARGET_SIZE}")
print(f"  Background slice every N : {BACKGROUND_SLICE_EVERY_N}")

## Shared Utility Functions

In [ ]:
def window_and_normalize(image_hu, wl=-600, ww=1500):
    """
    Applies CT lung windowing and normalizes to [0.0, 1.0] float32.
    Standard lung window: WL=-600, WW=1500  →  HU range [-1350, 150]
    """
    hu_min = wl - (ww / 2)
    hu_max = wl + (ww / 2)
    return np.clip((image_hu.astype(np.float32) - hu_min) / (hu_max - hu_min), 0.0, 1.0)


def normalize_uint8_image(image_uint8):
    """
    Normalizes a uint8 PNG image (0-255) to [0.0, 1.0] float32.
    Used for COVID slices that arrive as pre-windowed PNGs.
    """
    return (image_uint8.astype(np.float32) / 255.0)


def binarize_mask(mask, threshold=127):
    """
    Converts a grayscale mask to clean binary (0 or 1).
    Handles soft edges from JPEG compression artifacts in PNG masks.
    """
    return (mask > threshold).astype(np.uint8)


def world_to_voxel(world_coord, origin, spacing):
    """
    Converts physical mm coordinates to voxel indices.
    All inputs are (x, y, z) ordered as returned by SimpleITK.
    """
    return np.round(np.abs(world_coord - origin) / spacing).astype(int)


def save_pair(img_float32, mask_uint8, name):
    """Saves an image/mask pair as .npy files."""
    np.save(os.path.join(OUTPUT_IMG_DIR, name), img_float32)
    np.save(os.path_join(OUTPUT_MASK_DIR, name), mask_uint8)


# Fix the typo above
def save_pair(img_float32, mask_uint8, name):
    np.save(os.path.join(OUTPUT_IMG_DIR,  name), img_float32)
    np.save(os.path.join(OUTPUT_MASK_DIR, name), mask_uint8)


print("Utility functions ready.")

## Dataset 1 — COVID-19 CT Lung and Infection Segmentation

Both masks are **binary PNGs** (white = region, black = background), so `binarize_mask()` gives clean 0/1 values before class assignment.

**Mask build order (priority low → high):**
1. Background (default, 0)
2. Lung (class 1) — from `LungMasks/`
3. Infection (class 3) — from `InfectionMasks/`, overwrites lung where they overlap

In [ ]:
def process_covid_slices():
    print("Processing COVID-19 dataset (full, no slice limit)...")

    inf_mask_paths = sorted(glob.glob(os.path.join(COVID_INF_MASK_DIR, '*.*')))
    saved   = 0
    skipped = 0

    for inf_path in tqdm(inf_mask_paths):
        filename  = os.path.basename(inf_path)
        stem      = os.path.splitext(filename)[0]     # handles multi-dot names cleanly
        img_path  = os.path.join(COVID_IMG_DIR,       filename)
        lung_path = os.path.join(COVID_LUNG_MASK_DIR, filename)

        # ── Load ──
        img       = cv2.imread(img_path,  cv2.IMREAD_GRAYSCALE)
        inf_mask  = cv2.imread(inf_path,  cv2.IMREAD_GRAYSCALE)
        lung_mask = cv2.imread(lung_path, cv2.IMREAD_GRAYSCALE) if os.path.exists(lung_path) else None

        if img is None or inf_mask is None:
            skipped += 1
            continue

        # ── Resize ──
        img       = cv2.resize(img,      TARGET_SIZE, interpolation=cv2.INTER_LINEAR)
        inf_mask  = cv2.resize(inf_mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)
        if lung_mask is not None:
            lung_mask = cv2.resize(lung_mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)

        # ── FIX: Normalize image to [0,1] float32 — same scale as LUNA ──
        img_normalized = normalize_uint8_image(img)

        # ── Build unified 4-class mask ──
        unified_mask = np.zeros(TARGET_SIZE, dtype=np.uint8)  # all background

        # Layer 1: Lung (class 1)
        # LungMasks are binary PNGs: white=lung. Binarize to get clean 0/1.
        if lung_mask is not None:
            unified_mask[binarize_mask(lung_mask) == 1] = CLASS_LUNG

        # Layer 2: Infection (class 3) — overwrites lung pixels at overlap
        # InfectionMasks are also binary PNGs. Binarize before assigning class index.
        unified_mask[binarize_mask(inf_mask) == 1] = CLASS_INFECTION

        # ── Save ──
        save_pair(img_normalized, unified_mask, f"covid_{stem}.npy")
        saved += 1

    print(f"COVID done — saved: {saved:,}  |  skipped (missing file): {skipped}")


process_covid_slices()

## Dataset 2 — LUNA16

The `seg-lungs-LUNA16` folder contains **both** the raw CT `.mhd` files and their paired lung segmentation `.mhd` files. We separate them at runtime by checking SimpleITK's pixel type — the seg files are integer/label type while CT files are float/int16 HU values.

**Slice selection strategy:**
- All slices containing a nodule → always saved
- Non-nodule slices → saved every `N`th slice (balanced background sampling)

**Mask build order (priority low → high):**
1. Background (0)
2. Lung (class 1) — from seg `.mhd` file
3. Nodule (class 2) — from `annotations.csv`, drawn as 3D sphere, overwrites lung

In [ ]:
def separate_ct_and_seg_files(img_dir):
    """
    The seg-lungs-LUNA16 folder contains both raw CT volumes and their
    lung segmentation masks, all as .mhd files.

    Strategy: read pixel type via SimpleITK.
    - CT volumes    → pixel type sitkInt16 or sitkFloat32 (HU values, large range)
    - Seg masks     → pixel type sitkUInt8 or sitkInt16 (0/1 binary labels)

    We also match by filename: LUNA16 seg files are typically named with the
    same UID as the CT. We pair them by shared UID prefix and check which has
    binary content (max value <= 1).

    Returns:
        ct_to_seg : dict mapping ct_path -> seg_path (or None if no seg found)
    """
    all_mhd = sorted(glob.glob(os.path.join(img_dir, '*.mhd')))
    ct_files  = []
    seg_files = []

    print(f"Found {len(all_mhd)} .mhd files. Classifying CT vs seg...")

    for path in tqdm(all_mhd, desc='Classifying files'):
        try:
            img   = sitk.ReadImage(path)
            arr   = sitk.GetArrayFromImage(img)
            # CT: HU values span roughly -1500 to +3000
            # Seg: only 0 and 1 (binary lung mask)
            if arr.max() <= 1 and arr.min() >= 0:
                seg_files.append(path)
            else:
                ct_files.append(path)
        except Exception as e:
            print(f"  Warning: could not read {os.path.basename(path)}: {e}")

    print(f"  CT volumes : {len(ct_files)}")
    print(f"  Seg masks  : {len(seg_files)}")

    # Build a UID -> seg_path lookup (UIDs are the long alphanumeric filenames)
    seg_lookup = {}
    for seg_path in seg_files:
        uid = os.path.basename(seg_path).replace('.mhd', '')
        seg_lookup[uid] = seg_path

    ct_to_seg = {}
    unmatched = 0
    for ct_path in ct_files:
        uid = os.path.basename(ct_path).replace('.mhd', '')
        seg = seg_lookup.get(uid, None)
        ct_to_seg[ct_path] = seg
        if seg is None:
            unmatched += 1

    if unmatched > 0:
        print(f"  Warning: {unmatched} CT files have no matching seg mask.")

    return ct_to_seg


# Run the classification — do this once before the main loop
ct_to_seg = separate_ct_and_seg_files(LUNA_IMG_DIR)
print(f"\nReady to process {len(ct_to_seg)} CT volumes.")

In [ ]:
def process_luna16(ct_to_seg, annotations, n=BACKGROUND_SLICE_EVERY_N):
    """
    Processes all LUNA16 CT volumes.

    Slice selection:
      - Nodule slices      : always saved
      - Non-nodule slices  : saved every n-th slice (balanced background sampling)

    Mask layers (low to high priority):
      0 = background  (default)
      1 = lung        (from seg .mhd)
      2 = nodule      (drawn from annotations as 3D sphere, overwrites lung)
    """
    print(f"Processing LUNA16 — background slice every {n}th...")

    total_nodule_slices = 0
    total_bg_slices     = 0

    for ct_path, seg_path in tqdm(ct_to_seg.items()):
        patient_id = os.path.basename(ct_path).replace('.mhd', '')

        # ── Load CT ──
        itk_img   = sitk.ReadImage(ct_path)
        img_array = sitk.GetArrayFromImage(itk_img)   # (Z, Y, X), HU values
        origin    = np.array(itk_img.GetOrigin())     # (x, y, z) mm
        spacing   = np.array(itk_img.GetSpacing())    # (x, y, z) mm/voxel
        n_slices  = img_array.shape[0]

        # ── Load lung seg mask ──
        if seg_path is not None:
            itk_seg       = sitk.ReadImage(seg_path)
            lung_array_3d = (sitk.GetArrayFromImage(itk_seg) > 0).astype(np.uint8)  # (Z,Y,X) binary
        else:
            # Should not happen given our dataset, but safe fallback
            print(f"  No seg for {patient_id}, skipping.")
            continue

        # ── Window the whole volume once (faster than per-slice) ──
        img_windowed = window_and_normalize(img_array)  # (Z, Y, X), float32 [0,1]

        # ── Build 3D nodule mask from annotations ──
        nodule_mask_3d  = np.zeros_like(img_array, dtype=np.uint8)
        patient_nodules = annotations[annotations['seriesuid'] == patient_id]
        nodule_z_set    = set()

        for _, nodule in patient_nodules.iterrows():
            world_coord = np.array([nodule['coordX'], nodule['coordY'], nodule['coordZ']])
            diameter_mm = nodule['diameter_mm']

            v_x, v_y, v_z = world_to_voxel(world_coord, origin, spacing)
            radius_px     = max(1, int((diameter_mm / 2.0) / spacing[0]))

            z_min = max(0, v_z - radius_px);  z_max = min(n_slices,              v_z + radius_px + 1)
            y_min = max(0, v_y - radius_px);  y_max = min(img_array.shape[1],    v_y + radius_px + 1)
            x_min = max(0, v_x - radius_px);  x_max = min(img_array.shape[2],    v_x + radius_px + 1)

            # Vectorized sphere (replaces slow triple for-loop)
            zz, yy, xx = np.ogrid[z_min:z_max, y_min:y_max, x_min:x_max]
            sphere = ((zz-v_z)**2 + (yy-v_y)**2 + (xx-v_x)**2) <= radius_px**2
            nodule_mask_3d[z_min:z_max, y_min:y_max, x_min:x_max][sphere] = CLASS_NODULE

            for z in range(z_min, z_max):
                nodule_z_set.add(z)

        # ── Determine which slices to save ──
        slices_to_save = {}   # z_index -> 'nodule' or 'background'

        for z in range(n_slices):
            if z in nodule_z_set:
                slices_to_save[z] = 'nodule'
            elif z % n == 0:                  # every Nth non-nodule slice
                slices_to_save[z] = 'background'

        # ── Extract and save 2D slices ──
        for z_slice, slice_type in slices_to_save.items():

            # Image: already windowed, just resize
            img_resized = cv2.resize(
                img_windowed[z_slice], TARGET_SIZE, interpolation=cv2.INTER_LINEAR
            )

            # Unified mask
            unified_mask = np.zeros(img_array.shape[1:], dtype=np.uint8)

            # Layer 1: Lung (class 1) from seg file
            unified_mask[lung_array_3d[z_slice] > 0] = CLASS_LUNG

            # Layer 2: Nodule (class 2) overwrites lung where nodule exists
            unified_mask[nodule_mask_3d[z_slice] == CLASS_NODULE] = CLASS_NODULE

            mask_resized = cv2.resize(unified_mask, TARGET_SIZE, interpolation=cv2.INTER_NEAREST)

            save_name = f"luna_{patient_id}_z{z_slice:04d}.npy"
            save_pair(img_resized, mask_resized, save_name)

            if slice_type == 'nodule':
                total_nodule_slices += 1
            else:
                total_bg_slices += 1

    print(f"LUNA16 done.")
    print(f"  Nodule slices saved     : {total_nodule_slices:,}")
    print(f"  Background slices saved : {total_bg_slices:,}  (every {n}th non-nodule slice)")
    print(f"  Total LUNA slices       : {total_nodule_slices + total_bg_slices:,}")


try:
    annotations = pd.read_csv(LUNA_ANNOTATIONS)
    print(f"Loaded annotations.csv — {len(annotations)} nodule records.")
    process_luna16(ct_to_seg, annotations)
except FileNotFoundError:
    print(f"ERROR: annotations.csv not found at {LUNA_ANNOTATIONS}")

## Sanity Check — Visualize Random Samples

Run this after both pipelines complete to confirm:
- Images are grayscale [0,1] float32 from both datasets
- Masks show the correct class colors with no raw pixel values leaking in
- Infection (yellow) sits inside the lung (green) region as expected

In [ ]:
CLASS_COLORS = np.array([
    [0,   0,   0  ],   # 0 background  — black
    [0,   200, 0  ],   # 1 lung        — green
    [220, 50,  50 ],   # 2 nodule      — red
    [255, 220, 0  ],   # 3 infection   — yellow
], dtype=np.uint8)

CLASS_NAMES = {0: 'BG', 1: 'Lung', 2: 'Nodule', 3: 'Infection'}


def colorize_mask(mask):
    rgb = np.zeros((*mask.shape, 3), dtype=np.uint8)
    for cls_id, color in enumerate(CLASS_COLORS):
        rgb[mask == cls_id] = color
    return rgb


def visualize_samples(n=6, seed=42):
    all_imgs  = sorted(glob.glob(os.path.join(OUTPUT_IMG_DIR,  '*.npy')))
    all_masks = sorted(glob.glob(os.path.join(OUTPUT_MASK_DIR, '*.npy')))

    if not all_imgs:
        print("No files found. Run both pipelines first."); return

    # Show 3 COVID + 3 LUNA samples side by side
    covid_imgs = [p for p in all_imgs if os.path.basename(p).startswith('covid_')]
    luna_imgs  = [p for p in all_imgs if os.path.basename(p).startswith('luna_')]

    np.random.seed(seed)
    chosen = []
    if covid_imgs: chosen += list(np.random.choice(covid_imgs, min(3, len(covid_imgs)), replace=False))
    if luna_imgs:  chosen += list(np.random.choice(luna_imgs,  min(3, len(luna_imgs)),  replace=False))

    fig, axes = plt.subplots(len(chosen), 3, figsize=(12, len(chosen) * 4))
    if len(chosen) == 1: axes = [axes]

    for row, img_path in enumerate(chosen):
        mask_path = img_path.replace(OUTPUT_IMG_DIR, OUTPUT_MASK_DIR)
        img  = np.load(img_path)
        mask = np.load(mask_path)
        name = os.path.basename(img_path)

        # Verify pixel ranges
        assert img.dtype  == np.float32,  f"Image should be float32, got {img.dtype}"
        assert img.min()  >= 0.0,         f"Image min below 0: {img.min()}"
        assert img.max()  <= 1.0,         f"Image max above 1: {img.max()}"
        assert set(np.unique(mask)).issubset({0,1,2,3}), f"Unexpected class values: {np.unique(mask)}"

        mask_color = colorize_mask(mask)
        overlay    = np.stack([img]*3, axis=-1)
        blend      = np.clip(overlay * 0.6 + mask_color / 255.0 * 0.4, 0, 1)

        classes_present = ', '.join(CLASS_NAMES[c] for c in np.unique(mask))

        axes[row][0].imshow(img, cmap='gray', vmin=0, vmax=1)
        axes[row][0].set_title(f'{name}\n[{img.min():.3f}, {img.max():.3f}]  |  Classes: {classes_present}', fontsize=7)
        axes[row][0].axis('off')

        axes[row][1].imshow(mask_color)
        axes[row][1].set_title('Mask', fontsize=8)
        axes[row][1].axis('off')

        axes[row][2].imshow(blend)
        axes[row][2].set_title('Overlay', fontsize=8)
        axes[row][2].axis('off')

    plt.tight_layout()
    out_path = '/kaggle/working/sample_verification.png'
    plt.savefig(out_path, dpi=120, bbox_inches='tight')
    plt.show()
    print(f"Assertions passed. Saved to {out_path}")


visualize_samples()

## Dataset Statistics + Class Weights

Run this before training. The class weights printed at the end go directly into your `CrossEntropyLoss`.

In [ ]:
def compute_dataset_stats():
    all_masks = sorted(glob.glob(os.path.join(OUTPUT_MASK_DIR, '*.npy')))
    if not all_masks:
        print("No masks found."); return

    pixel_counts   = np.zeros(NUM_CLASSES, dtype=np.int64)
    slice_counts   = np.zeros(NUM_CLASSES, dtype=np.int64)
    covid_n, luna_n = 0, 0

    for mask_path in tqdm(all_masks, desc='Computing stats'):
        mask = np.load(mask_path)
        name = os.path.basename(mask_path)
        if name.startswith('covid_'): covid_n += 1
        elif name.startswith('luna_'): luna_n  += 1
        for cls in range(NUM_CLASSES):
            px = int(np.sum(mask == cls))
            pixel_counts[cls] += px
            if px > 0: slice_counts[cls] += 1

    total = pixel_counts.sum()
    names = ['Background', 'Lung', 'Nodule', 'Infection']

    print(f"\n{'='*60}")
    print(f" Total slices : {len(all_masks):,}  (COVID: {covid_n:,}  |  LUNA: {luna_n:,})")
    print(f"{'='*60}")
    print(f"{'Class':<15} {'Pixels':>12} {'% of total':>11} {'Slices w/ class':>18}")
    print(f"{'-'*60}")
    for cls in range(NUM_CLASSES):
        pct = 100.0 * pixel_counts[cls] / total if total > 0 else 0
        print(f"{names[cls]:<15} {pixel_counts[cls]:>12,} {pct:>10.2f}%  {slice_counts[cls]:>15,}")
    print(f"{'='*60}")

    # Median-frequency balancing (more stable than raw inverse frequency)
    freq          = pixel_counts.astype(np.float64) / total
    freq          = np.where(freq == 0, 1e-8, freq)
    median_freq   = np.median(freq)
    class_weights = median_freq / freq
    # Cap extreme weights (nodules can be <0.1% of pixels → weight explodes)
    class_weights = np.clip(class_weights, 0.1, 20.0)

    print("\nSuggested class weights (median-frequency balancing, capped at 20):")
    weight_str = []
    for cls in range(NUM_CLASSES):
        print(f"  Class {cls} ({names[cls]:<12}): {class_weights[cls]:.4f}")
        weight_str.append(f"{class_weights[cls]:.4f}")

    print("\nCopy this into your training script:")
    print(f"  weights = torch.tensor([{', '.join(weight_str)}], dtype=torch.float32).to(device)")
    print(f"  criterion = nn.CrossEntropyLoss(weight=weights)")


compute_dataset_stats()